# Pipeline — Paso 8: Selección Top-35 glosas

Selecciona las 35 señas con mayor número de muestras verificadas y genera dataset_top35.csv y dataset_keypoints.csv.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
from pathlib import Path

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))
from utils.config import (DATA_DIR, KEYPOINTS_DIR, N_FRAMES, N_KEYPOINTS,
                           CSV_TOP35, CSV_KEYPOINTS, CSV_LIMPIO)

TOP_N = 35   # número de glosas a seleccionar


def main():
    if not os.path.exists(CSV_LIMPIO):
        print(f"[ERROR] No existe {CSV_LIMPIO}")
        print("  Ejecuta primero: python pipeline/06_limpiar_dataset.py")
        return

    df = pd.read_csv(CSV_LIMPIO)

    # ── 1. Filtrar solo Label Studio ─────────────────────────────────────────
    df_ls = df[df["herramienta"] == "label_studio"].copy()
    print(f"Registros Label Studio: {len(df_ls)}  |  glosas únicas: {df_ls['glosa'].nunique()}")

    # ── 2. Verificar qué keypoints existen realmente ─────────────────────────
    def tiene_kp(video_id):
        p = os.path.join(KEYPOINTS_DIR, f"{video_id}.npy")
        if not os.path.exists(p):
            return False
        try:
            arr = np.load(p)
            return arr.shape == (N_FRAMES, N_KEYPOINTS)
        except Exception:
            return False

    print("Verificando keypoints existentes...")
    df_ls["npy_path"]     = df_ls["video_id"].apply(
        lambda v: os.path.join(KEYPOINTS_DIR, f"{v}.npy"))
    df_ls["kp_existe"]    = df_ls["video_id"].apply(tiene_kp)

    total_kp  = df_ls["kp_existe"].sum()
    sin_kp    = (~df_ls["kp_existe"]).sum()
    print(f"  Con keypoint válido : {total_kp}")
    print(f"  Sin keypoint (SKIP) : {sin_kp}")

    if total_kp == 0:
        print("\n[ERROR] No hay keypoints extraídos en:", KEYPOINTS_DIR)
        print("  Ejecuta primero: python pipeline/07_extraer_keypoints.py")
        return

    df_con_kp = df_ls[df_ls["kp_existe"]].copy()

    # ── 3. Seleccionar las top-N glosas por cantidad de muestras ─────────────
    conteo = df_con_kp["glosa"].value_counts()
    print(f"\nDistribución de muestras (glosas con keypoint):")
    print(f"  max={conteo.max()}  min={conteo.min()}  media={conteo.mean():.1f}  "
          f"glosas_totales={len(conteo)}")

    top_n = min(TOP_N, len(conteo))
    glosas_top = conteo.head(top_n).index.tolist()
    print(f"\nTop {top_n} glosas seleccionadas (por nº de muestras):")
    for i, g in enumerate(glosas_top, 1):
        print(f"  {i:>2}. {g:<35} {conteo[g]} muestras")

    df_top = df_con_kp[df_con_kp["glosa"].isin(glosas_top)].copy()
    print(f"\nRegistros en top-{top_n}: {len(df_top)}")

    # ── 4. Guardar CSVs ──────────────────────────────────────────────────────
    # dataset_top35.csv  (sin columna kp_existe, la lee paso 12)
    df_top.drop(columns=["kp_existe"], errors="ignore").to_csv(
        CSV_TOP35, index=False, encoding="utf-8-sig")

    # dataset_keypoints.csv (mismo contenido, lo lee paso 11)
    df_top.drop(columns=["kp_existe"], errors="ignore").to_csv(
        CSV_KEYPOINTS, index=False, encoding="utf-8-sig")

    print(f"\nGuardado: {CSV_TOP35}")
    print(f"Guardado: {CSV_KEYPOINTS}")
    print(f"\n=== RESUMEN ===")
    print(f"Glosas : {top_n}")
    print(f"Videos : {len(df_top)}")
    print(f"Media  : {len(df_top)/top_n:.1f} videos/glosa")

main()


C:\Users\dell\AppData\Local\Temp\ipykernel_8432\3968758621.py:12: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


Registros Label Studio: 911  |  glosas únicas: 503
Verificando keypoints existentes...


  Con keypoint válido : 911
  Sin keypoint (SKIP) : 0

Distribución de muestras (glosas con keypoint):
  max=8  min=1  media=1.8  glosas_totales=503

Top 35 glosas seleccionadas (por nº de muestras):
   1. AH,CLARO                            8 muestras
   2. ENCENDIDO                           7 muestras
   3. CORRECTO                            5 muestras
   4. EL AÑO PASADO                       5 muestras
   5. ESCUCHAR+(1h)RUMBLE                 5 muestras
   6. FAMOSO                              5 muestras
   7. FUEGOS ARTIFICIALES                 5 muestras
   8. MUDARSE                             5 muestras
   9. CRECER                              5 muestras
  10. DEJAR                               4 muestras
  11. CUATRO DÓLARES                      4 muestras
  12. AUDIOLOGÍA+AGENTE                   4 muestras
  13. GUITARRA                            4 muestras
  14. AVERGONZAR                          4 muestras
  15. AVISO                               4 muestras
  16.